# Style Transfer Training on Kaggle

**Before running any cell**, do these two things in the Kaggle notebook UI (no code required):

1. **Add COCO 2017 dataset**  
   Notebook sidebar → *+ Add Data* → search **`awsaf49/coco-2017-dataset`** → Add.  
   This mounts the dataset at `/kaggle/input/datasets/awsaf49/coco-2017-dataset/` — no download needed.

2. **Enable GPU**  
   Notebook sidebar → *Settings* → Accelerator → **GPU T4 x1**.

Then run the cells below top-to-bottom.


## Step 1 — Verify GPU and COCO images are available

In [ ]:
import os
import pathlib

# ── GPU check ────────────────────────────────────────────────────────────────
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ── COCO images check ────────────────────────────────────────────────────────
COCO_TRAIN = pathlib.Path("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017")
imgs = list(COCO_TRAIN.glob("*.jpg"))
print(f"COCO train images found: {len(imgs):,}")   # expect ~118,287
assert len(imgs) > 1000, "Dataset not mounted — add awsaf49/coco-2017-dataset in notebook settings"


## Step 1b — Verify internet access

**Internet must be enabled** for training to work — VGG16 weights (~550 MB) are downloaded on the first run.

Enable it in: *Notebook sidebar → Settings → Internet → On*  
Then re-run this cell. If the check passes, continue to Step 2.

In [ ]:
import urllib.request

try:
    urllib.request.urlopen("https://download.pytorch.org", timeout=5)
    print("✓ Internet is available — safe to continue.")
except Exception:
    raise RuntimeError(
        "\n\n"
        "  ✗ No internet access detected.\n\n"
        "  Training requires internet to download VGG16 weights (~550 MB).\n"
        "  Fix: Notebook sidebar → Settings → Internet → On\n"
        "  Then re-run this cell before proceeding.\n"
    )

## Step 2 — Clone the repo and install trainer dependencies

In [ ]:
import subprocess, sys, os

REPO_URL = "https://github.com/PeterWazinski/style_transfer.git"
REPO_DIR = pathlib.Path("/kaggle/working/style_transfer")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

os.chdir(REPO_DIR)

# Install trainer deps (torch is already present on Kaggle GPU images)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[trainer]"], check=True)
print("Install done. Working dir:", os.getcwd())


## Step 3 — Upload your style reference image

Upload the style image you want to train via the Kaggle notebook UI:
*File browser (left panel) → Upload → select your JPEG*.

It will land at `/kaggle/working/<filename>`. Set `STYLE_IMAGE` below accordingly.

In [ ]:
# ── Edit these three lines to match your job ─────────────────────────────────
STYLE_IMAGE  = pathlib.Path("/kaggle/working/my_style.jpg")   # ← your uploaded file
STYLE_ID     = "my_style"     # slug used as folder name and catalog ID
STYLE_NAME   = "My Style"     # display name shown in the gallery
# ─────────────────────────────────────────────────────────────────────────────

assert STYLE_IMAGE.exists(), f"Style image not found: {STYLE_IMAGE}\nUpload it via the file browser first."
print("Style image ready:", STYLE_IMAGE)


## Step 4 — Run training

Typical wall-clock times on a Kaggle T4:
| Epochs | COCO images | ~Time |
|---|---|---|
| 1 | 118 k | ~1.5 h |
| 2 | 118 k | ~3 h |

Kaggle gives you **30 h/week** GPU, so 2 epochs comfortably fits in one session.

In [ ]:
import os as _os

OUT_DIR = REPO_DIR / "styles" / STYLE_ID
CONTENT_IMAGE = pathlib.Path("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/val2017") \
    / "000000000139.jpg"   # a single COCO val image used for the preview thumbnail

cmd = [
    sys.executable, str(REPO_DIR / "main_style_trainer.py"), "train",
    "--style",          str(STYLE_IMAGE),
    "--coco",           str(COCO_TRAIN),        # 118 k JPEG images
    "--out",            str(OUT_DIR),
    "--id",             STYLE_ID,
    "--name",           STYLE_NAME,
    "--content",        str(CONTENT_IMAGE),      # generates preview.jpg after training
    "--device",         "cuda",                  # T4 GPU
    "--epochs",         "2",
    "--batch-size",     "4",
    "--image-size",     "256",
]

# Ensure src/ is importable inside the subprocess (editable install alone is
# not always enough on Kaggle's Python 3.12 kernel).
env = _os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR) + _os.pathsep + env.get("PYTHONPATH", "")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=str(REPO_DIR), env=env)
print("\nTraining complete. Output files:")
for f in OUT_DIR.iterdir():
    print(" ", f.name, f"({f.stat().st_size // 1024} KB)")


## Step 5 — Check preview and download the model

After training finishes you should have `model.onnx`, `model.pth`, and `preview.jpg` in the output directory.

Download everything via: Kaggle file browser → right-click on the output folder → *Download*.

Or zip and use the cell below to make a single download link.

In [ ]:
import shutil, zipfile

# --- show preview -----------------------------------------------------------
from IPython.display import Image as IPImage, display
preview = OUT_DIR / "preview.jpg"
if preview.exists():
    display(IPImage(str(preview)))
else:
    print("preview.jpg not found — training may not have completed cleanly")

# --- list output files -------------------------------------------------------
print("\nOutput files:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name:30s}  {f.stat().st_size // 1024:6d} KB")

# --- copy to /kaggle/output/ so Kaggle makes them downloadable ---------------
DOWNLOAD_DIR = pathlib.Path("/kaggle/output") / STYLE_ID
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
for f in OUT_DIR.iterdir():
    shutil.copy2(f, DOWNLOAD_DIR / f.name)

# --- also zip into a single file for convenience -----------------------------
zip_path = pathlib.Path("/kaggle/output") / f"{STYLE_ID}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(DOWNLOAD_DIR.iterdir()):
        zf.write(f, f.name)

print(f"\nZip: {zip_path}  ({zip_path.stat().st_size // 1024} KB)")
print("Download from the Output tab (right panel) in the Kaggle notebook UI.")
